# Notes

for Mongabay discussion

In [6]:
import pandas
import numpy
import pygsheets
import datetime
#import re
import pytz

In [2]:
# define the excel file to save tables in
current_time = datetime.datetime.now(pytz.timezone('US/Eastern')).strftime("%Y-%m-%d_T%H%M%S")

## import data

In [69]:
#fuel_type = 'Gas'
fuel_type = 'Oil'
#fuel_type = 'NGL'

In [70]:
gc = pygsheets.authorize(service_account_env_var='GDRIVE_API_CREDENTIALS')
spreadsheet = gc.open_by_key('1foPLE6K-uqFlaYgLPAUxzeXfDO5wOOqE7tibNHeqTek') # CURRENT sheet
#spreadsheet = gc.open_by_key('1MK6yVWDUnbfXRjynMQcxIaeUc666XIKeKdVwtK1bd8I') # mar 2025 oil release

gas_pipes = spreadsheet.worksheet('title', 'Gas pipelines').get_as_df(start='A3')
oil_pipes = spreadsheet.worksheet('title', 'Oil/NGL pipelines').get_as_df(start='A3')

gas_pipes = gas_pipes.drop('WKTFormat', axis=1) # delete WKTFormat column
oil_pipes = oil_pipes.drop('WKTFormat', axis=1)
#pipes_df_orig = gas_pipes.copy() #pandas.concat([oil_pipes, gas_pipes], ignore_index=True)

#get other relevant sheets
country_ratios_df = spreadsheet.worksheet('title', 'Country ratios by pipeline').get_as_df()
owners_df_orig = spreadsheet.worksheet('title', 'Pipeline operators/owners').get_as_df(start='A2')
# country_ratios_df = country_ratios_df.loc[country_ratios_df.Wiki!='']

# remove empty cells for pipes, owners
if fuel_type=='Gas':
    pipes_df_orig = gas_pipes.copy()
if fuel_type in ['Oil','NGL']:
    pipes_df_orig = oil_pipes.copy()
pipes_df_orig = pipes_df_orig.loc[pipes_df_orig['PipelineName']!='']
#pipes_df_orig = pipes_df_orig.loc[pipes_df_orig['Wiki']!='']
pipes_df_orig = pipes_df_orig.loc[pipes_df_orig.Fuel.isin(fuel_options)]

owners_df_orig = owners_df_orig.loc[owners_df_orig['PipelineName']!='']
#owners_df_orig = owners_df_orig.loc[owners_df_orig['Wiki']!='']
owners_df_orig = owners_df_orig.loc[owners_df_orig.Status!='N/A']
owners_df_orig.set_index('ProjectID', inplace=True)

In [71]:
country_ratios_df = country_ratios_df.replace('--', numpy.nan)
owners_df_orig = owners_df_orig.replace('',numpy.nan)
owners_df_orig = owners_df_orig.replace('--',numpy.nan)
pipes_df_orig = pipes_df_orig.replace('--',numpy.nan)

/var/folders/fl/t07mc8053p33mn6mdmvp45580000gn/T/ipykernel_18941/1118424270.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  country_ratios_df = country_ratios_df.replace('--', numpy.nan)
/var/folders/fl/t07mc8053p33mn6mdmvp45580000gn/T/ipykernel_18941/1118424270.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  owners_df_orig = owners_df_orig.replace('--',numpy.nan)
/var/folders/fl/t07mc8053p33mn6mdmvp45580000gn/T/ipykernel_18941/1118424270.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be remo

In [72]:
region_df_orig = spreadsheet.worksheet('title', 'Country dictionary').get_as_df(start='A2')

region_name = 'Global'; region_df_touse = region_df_orig.copy()
#region_name = 'AsiaGasTracker'; region_df_touse = region_df_orig.loc[region_df_orig.AsiaGasTracker=='Yes']
#region_name = 'EuroGasTracker'; region_df_touse = region_df_orig.loc[region_df_orig.EuroGasTracker=='Yes']
#region_name = 'AfricaGasTracker'; region_df_touse = region_df_orig.loc[region_df_orig.AfricaGasTracker=='Yes']
#region_name = 'LatinAmericaTracker'; region_df_touse = region_df_orig.loc[region_df_orig.LatinAmericaTracker=='Yes']
#region_df_agt.copy()

#region_df_touse = region_df_orig.copy()

In [73]:
region_df_touse_cleaned = region_df_touse.loc[(region_df_touse.Region!='--')&
                                            (region_df_touse.SubRegion!='--')]
multiindex_region_subregion = region_df_touse_cleaned.groupby(['Region','SubRegion'])['Country'].count().index
multiindex_region_subregion

MultiIndex([(  'Africa',                 'Northern Africa'),
            (  'Africa',              'Sub-Saharan Africa'),
            ('Americas', 'Latin America and the Caribbean'),
            ('Americas',                'Northern America'),
            (    'Asia',                    'Central Asia'),
            (    'Asia',                    'Eastern Asia'),
            (    'Asia',              'South-eastern Asia'),
            (    'Asia',                   'Southern Asia'),
            (    'Asia',                    'Western Asia'),
            (  'Europe',                  'Eastern Europe'),
            (  'Europe',                 'Northern Europe'),
            (  'Europe',                 'Southern Europe'),
            (  'Europe',                  'Western Europe'),
            ( 'Oceania',       'Australia and New Zealand'),
            ( 'Oceania',                       'Melanesia'),
            ( 'Oceania',                      'Micronesia'),
            ( 'Oceania',

In [74]:
country_ratios_df.columns

Index(['PipelineName', 'SegmentName', 'ProjectID', 'Country',
       'LengthEstimateKmByCountry', 'LengthPerCountryFraction', 'Region',
       'SubRegion', 'RegionOld', 'PipelineBubbleRegion', 'Wiki', 'Status',
       'Fuel', 'CapacityBOEd', 'CapacityBOEdPerCountry', 'FID', 'CostUSD',
       'CostEuro', 'LengthKnownKm', 'LengthEstimateKm',
       'LengthMergedKmByPipeline', 'LengthKnownKmByCountry',
       'LengthMergedKmByCountry', 'CostUSDPerKm', 'CostEuroPerKm', 'Owner',
       'Parent', 'H2Status', 'H2Type', 'CancelledYear', 'ProposalYear',
       'ConstructionYear', 'ShelvedYear', 'StartYearEarliest', 'StartCountry',
       'EndCountry', 'RouteType', 'RouteAccuracy'],
      dtype='object')

## proposed, construction, shelved

### grouped by ProjectID

In [78]:
filtered_df = country_ratios_df.loc[
    (country_ratios_df.Fuel == 'Oil') &
    (country_ratios_df.Status.isin(['construction', 'proposed', 'shelved']))
].groupby('ProjectID').agg({
    'PipelineName': lambda x: ', '.join(x.unique()), # or 'unique', 'min', 'max', 'last', 'join', etc.
    'SegmentName': lambda x: ', '.join(map(str, x.unique())),
    'Wiki': lambda x: ', '.join(x.unique()),
    'LengthMergedKmByCountry': 'sum',
    'Country': lambda x: ', '.join(x.unique()),
    'Region': lambda x: ', '.join(x.unique()),
    'SubRegion': lambda x: ', '.join(x.unique()),
    'Fuel': lambda x: ', '.join(x.unique()),
    'Status': lambda x: ', '.join(x.unique()),
    'RouteType': lambda x: ', '.join(x.unique()),
    'RouteAccuracy': lambda x: ', '.join(x.unique())
}).sort_values('LengthMergedKmByCountry', ascending=False)

filtered_df.to_excel(current_time+'_proposed_shelved_construction_summed_by_length-groupby_projectid.xlsx')
filtered_df

,PipelineName,SegmentName,Wiki,LengthMergedKmByCountry,Country,Region,SubRegion,Fuel,Status,RouteType,RouteAccuracy
ProjectID,,,,,,,,,,,
P3844,Canadian Prosperity Project,,https://www.gem.wiki/Canadian_Prosperity_Project,4800.0,Canada,Americas,Northern America,Oil,shelved,Mapped route (at any accuracy),no route
P2481,Tazama Oil Pipeline,Expansion,https://www.gem.wiki/Tazama_Oil_Pipeline,1710.0,"Zambia, Tanzania",Africa,Sub-Saharan Africa,Oil,proposed,Mapped route (at any accuracy),high
P3843,Paradip Numaligarh Crude Pipeline (PNCPL),,https://www.gem.wiki/Paradip_Numaligarh_Crude_...,1630.0,India,Asia,Southern Asia,Oil,construction,Mapped route (at any accuracy),high
P3750,Western Crude Oil Parallel Pipeline,Golmud Branch,https://www.gem.wiki/Western_Crude_Oil_Paralle...,1620.0,China,Asia,Eastern Asia,Oil,proposed,Mapped route (at any accuracy),low
P0541,East African Crude Oil Pipeline (EACOP),,https://www.gem.wiki/East_African_Crude_Oil_Pi...,1443.0,"Tanzania, Uganda",Africa,Sub-Saharan Africa,Oil,proposed,Mapped route (at any accuracy),high
...,...,...,...,...,...,...,...,...,...,...,...
P5204,Waidiao-Cezi Oil Pipeline,Pipeline 1,https://www.gem.wiki/Waidiao-Cezi_Oil_Pipeline,3.5,China,Asia,Eastern Asia,Oil,proposed,Mapped route (at any accuracy),low
P6426,Zhuangxi Oil Pipeline Expansion Project 2,,,2.4,China,Asia,Eastern Asia,Oil,proposed,Mapped route (at any accuracy),no route
P6437,Dongying Shenchi Crude Oil Pipeline Phase III,,,1.5,China,Asia,Eastern Asia,Oil,proposed,Mapped route (at any accuracy),low


### grouped by PipelineName

In [79]:
filtered_df = country_ratios_df.loc[
    (country_ratios_df.Fuel == 'Oil') &
    (country_ratios_df.Status.isin(['construction', 'proposed', 'shelved']))
].groupby('PipelineName').agg({
    'ProjectID': lambda x: ', '.join(x.unique()), # or 'unique', 'min', 'max', 'last', 'join', etc.
    'SegmentName': lambda x: ', '.join(map(str, x.unique())),
    'Wiki': lambda x: ', '.join(x.unique()),
    'LengthMergedKmByCountry': 'sum',
    'Country': lambda x: ', '.join(x.unique()),
    'Region': lambda x: ', '.join(x.unique()),
    'SubRegion': lambda x: ', '.join(x.unique()),
    'Fuel': lambda x: ', '.join(x.unique()),
    'Status': lambda x: ', '.join(x.unique()),
    'RouteType': lambda x: ', '.join(x.unique()),
    'RouteAccuracy': lambda x: ', '.join(x.unique())
}).sort_values('LengthMergedKmByCountry', ascending=False)

filtered_df.to_excel(current_time+'_proposed_shelved_construction_summed_by_length-groupby_pipelinename.xlsx')
filtered_df

,ProjectID,SegmentName,Wiki,LengthMergedKmByCountry,Country,Region,SubRegion,Fuel,Status,RouteType,RouteAccuracy
PipelineName,,,,,,,,,,,
Canadian Prosperity Project,P3844,,https://www.gem.wiki/Canadian_Prosperity_Project,4800.00,Canada,Americas,Northern America,Oil,shelved,Mapped route (at any accuracy),no route
Basra–North Baghdad–Syria Oil Pipeline,"P3874, P3875","Phase I (Basra–Haditha Pipeline), Phase II (Ha...",https://www.gem.wiki/Basra%E2%80%93North_Baghd...,2495.24,"Iraq, Syria",Asia,Western Asia,Oil,proposed,Mapped route (at any accuracy),low
Kirkuk-Baniyas Oil Pipelines,"P5277, P5278, P5279","Rehabilitation Pipeline, Heavy Oil Pipeline, L...",https://www.gem.wiki/Kirkuk-Baniyas_Oil_Pipelines,2400.21,"Iraq, Syria",Asia,Western Asia,Oil,proposed,Mapped route (at any accuracy),medium
Lamu Port-South Sudan (LAPSSET) Pipeline,"P0531, P0538","Lokichar–Lamu Oil Pipeline Segment, South Suda...",https://www.gem.wiki/Lamu_Port-South_Sudan_(LA...,1928.26,"Kenya, South Sudan",Africa,Sub-Saharan Africa,Oil,proposed,Mapped route (at any accuracy),high
Tazama Oil Pipeline,P2481,Expansion,https://www.gem.wiki/Tazama_Oil_Pipeline,1710.00,"Zambia, Tanzania",Africa,Sub-Saharan Africa,Oil,proposed,Mapped route (at any accuracy),high
...,...,...,...,...,...,...,...,...,...,...,...
Liuheng Island Connection Line,P6293,,,7.00,China,Asia,Eastern Asia,Oil,proposed,Mapped route (at any accuracy),low
Shulanghu Island - Liutiaoxi Oil Pipeline,P6294,,,7.00,China,Asia,Eastern Asia,Oil,proposed,Mapped route (at any accuracy),low
Zhuangxi Oil Pipeline Expansion Project 2,P6426,,,2.40,China,Asia,Eastern Asia,Oil,proposed,Mapped route (at any accuracy),no route


In [80]:
current_time

'2025-04-07_T142330'